# Understanding `MoleculeEnv` with simple and complex examples

This notebook explains every method in `MoleculeEnv` through two running examples:

1. **Simple example:** one carbon atom and one oxygen atom. This makes the pair tables, valid actions, merging, partial bond breakage, and splitting easy to inspect.
2. **Complex example:** a four-carbon chain plus an oxygen atom. We close a carbon ring, remove a non-bridge bond, merge oxygen into the molecule, and then remove a bridge to split the molecule.

A leading underscore, as in `_split_molecule`, means that the method is an internal helper. Normally, model code should call `step`, `bond_formation`, or `bond_breakage` and let the environment invoke these helpers.

## The four kinds of graph information

`MoleculeEnv` keeps four related views of the reaction system:

- **Node information:** one feature row per atom.
- **Current-edge information:** only bonds that currently exist; these become message-passing edges.
- **Candidate-pair information:** one fixed row for every unordered atom pair, including pairs with bond order zero.
- **Molecule information:** one feature row per connected component (subgraph).

The atom inventory and candidate-pair rows do not change during an episode. Bond orders and molecule membership do change.

In [25]:
from pathlib import Path
import sys

# Work whether Jupyter starts in codeBase/, src/, or a parent directory.
search_roots = [Path.cwd(), *Path.cwd().parents]
candidate_directories = []
for root in search_roots:
    candidate_directories.extend((root, root / "src"))
src_directory = next(
    (directory for directory in candidate_directories
     if (directory / "obj_graph.py").is_file()),
    None,
)
if src_directory is None:
    raise FileNotFoundError("Could not locate src/obj_graph.py.")

if str(src_directory) not in sys.path:
    sys.path.insert(0, str(src_directory))

import torch
from obj_edge import Bond
from obj_graph import MoleculeEnv
from obj_node import atom
from obj_subgraph import molecule

print("Using source files from:", src_directory)
print("PyTorch version:", torch.__version__)

Using source files from: /Users/boyuanyu/Documents/research/GNN/GNN_discover_species/codeBase/src
PyTorch version: 2.12.0


## Small notebook helpers

These functions are only for constructing and displaying examples. They are deliberately kept outside `MoleculeEnv`: the environment begins from already constructed molecule snapshots.

In [26]:
def singleton(index, element):
    """Construct the one-atom molecule used for an unbonded atom."""
    return molecule([atom(index, element)])


def connected_molecule(atom_specs, bond_specs):
    """Construct a chemically consistent molecule for an initial state.

    atom_specs: [(index, element), ...]
    bond_specs: [(index_i, index_j, bond_order), ...]
    """
    atoms_by_index = {index: atom(index, element) for index, element in atom_specs}
    bonds = []
    for index_i, index_j, order in bond_specs:
        node_i = atoms_by_index[index_i]
        node_j = atoms_by_index[index_j]
        node_i.form_bond(order)
        node_j.form_bond(order)
        bonds.append(Bond(node_i, node_j, order))
    species = molecule(
        atoms=sorted(atoms_by_index.values(), key=lambda member: member.index),
        bonds=bonds,
    )
    return species, atoms_by_index


def component_sets(env):
    """Return molecule membership as sorted atom-index tuples."""
    return [tuple(member.index for member in species.atoms) for species in env.molecules]


def pair_rows(env):
    """Return the two bond-order tables as easy-to-read rows."""
    return [
        {
            "pair": pair,
            "current_BO": env.current_BO[row],
            "maximum_BO": env.maximum_BO[row],
        }
        for row, pair in enumerate(env.pair_index)
    ]


def show_state(env, title):
    print(f"\n{title}")
    print("  components:", component_sets(env))
    print("  bonds:", {key: bond.order for key, bond in sorted(env.bonds.items())})
    print("  remaining valence:", {i: a.remaining_valence for i, a in sorted(env.atoms.items())})
    print("  pair tables:")
    for row in pair_rows(env):
        print("   ", row)


def show_shapes(observation):
    for name, value in observation.items():
        print(f"  {name:22s} shape={tuple(value.shape)} dtype={value.dtype}")


def show_expected_error(label, operation):
    try:
        operation()
    except (TypeError, ValueError, RuntimeError) as error:
        print(f"{label}: {type(error).__name__}: {error}")

## Method map

The notebook calls all public methods and observes every internal helper in the transition where it matters.

| Method | Responsibility | Simple example | Complex example |
|---|---|---|---|
| `__init__` | Register atoms/bonds and build caches | C and O singletons | carbon chain plus O |
| `step` | Dispatch one action and return the RL-style transition tuple | form C–O | all four complex transitions |
| `bond_formation` | Increase bond order and possibly merge components | directly form C=O | close ring and attach O |
| `bond_breakage` | Decrease bond order and possibly split a component | C=O → C–O → C + O | non-bridge removal and bridge removal |
| `get_valid_actions` | Give the maximum legal change per pair/action type | one pair | ten pairs with changing valences |
| `get_valid_action_masks` | Convert maximum changes to `(P, 3)` masks | one mask row | ten mask rows |
| `get_gnn_observation` | Build node, edge, pair, and molecule tensors | disconnected and bonded C/O | multiple changing components |
| `molecule_containing` | Look up the component containing an exact atom object | before/after C–O merge | before/after complex splits |
| `validate_state` | Check all invariants and caches | after simple edits | after complex sequence |
| `_replace_molecules` | Replace stale component snapshots | merge/split C and O | every topology edit |
| `_merge_molecules` | Build one component from two plus a new bond | forming C=O | attaching O to carbon |
| `_split_molecule` | Recompute components after complete bond removal | C–O removal makes two singletons | distinguish non-bridge from bridge removal |
| `_rebuild_atom_to_molecule` | Refresh atom-to-component lookup | after simple merge/split | after each component replacement |
| `_build_pair_tables` | Construct fixed pair rows at initialization | one C/O row | ten rows for five atoms |
| `_update_pair_tables` | Refresh affected pair rows after an action | the only row | rows incident to either endpoint |
| `_current_bond_order` | Return current order, using zero for no bond | C/O before and after formation | selected complex pairs |
| `_maximum_reachable_order` | Apply endpoint valence and order limit | C/O maximum is 2 | maximums change around edited atoms |
| `_require_registered_atom` | Reject foreign objects, even with a reused index | cloned carbon is rejected | protects every complex action |
| `_validate_bond_change` | Require a positive non-Boolean integer | 2 accepted, 0 rejected | protects multi-order changes |
| `_bond_key` | Canonicalize an unordered pair | `(1, 0)` becomes `(0, 1)` | keeps bond dictionary keys stable |
| `_bond_sort_key` | Sort bonds reproducibly | one bond | several complex bonds |
| `_minimum_atom_index` | Sort molecule rows reproducibly | components start at 0 and 1 | components start at their lowest atom ID |

# Part 1 — Simple C/O example

We begin with two singleton molecules. There are `N = 2` atoms, no current bonds, `M = 2` molecules, and `P = N(N-1)/2 = 1` candidate pair.

In [27]:
carbon_species = singleton(0, "C")
oxygen_species = singleton(1, "O")
carbon = carbon_species.atoms[0]
oxygen = oxygen_species.atoms[0]

simple_env = MoleculeEnv([carbon_species, oxygen_species], max_steps=10)
show_state(simple_env, "State created by __init__")
print("  atom_to_molecule:", {i: tuple(a.index for a in m.atoms)
                                  for i, m in simple_env.atom_to_molecule.items()})


State created by __init__
  components: [(0,), (1,)]
  bonds: {}
  remaining valence: {0: 4, 1: 2}
  pair tables:
    {'pair': (0, 1), 'current_BO': 0, 'maximum_BO': 2}
  atom_to_molecule: {0: (0,), 1: (1,)}


### `__init__`, `_build_pair_tables`, `_rebuild_atom_to_molecule`, and `_minimum_atom_index`

`__init__` treats the supplied molecules as the initial oven state. It registers the canonical atom and bond objects, sorts molecule rows by their minimum atom index, builds the atom-to-molecule lookup, creates the fixed unordered-pair rows, computes the first valid-action cache, and calls `validate_state`.

For two atoms, `_build_pair_tables` creates exactly one row: `(0, 1)`. `_rebuild_atom_to_molecule` maps atom 0 to `(0,)` and atom 1 to `(1,)`. `_minimum_atom_index` makes this component order reproducible.

### `get_valid_actions` and `get_valid_action_masks`

The action dictionary stores only the **maximum** legal positive change. If the maximum formation change is 2, both `bond_change=1` and `bond_change=2` are legal. The masks make those individual choices explicit: column 0 means change by 1, column 1 means change by 2, and column 2 means change by 3.

In [28]:
simple_actions = simple_env.get_valid_actions()
formation_mask, breakage_mask = simple_env.get_valid_action_masks()

print("valid_actions:", simple_actions)
print("formation_mask:", formation_mask.tolist())
print("breakage_mask: ", breakage_mask.tolist())
print("Interpretation: C and O may form a single or double bond, but no bond exists to break.")

valid_actions: {(0, 1, 'formation'): 2}
formation_mask: [[True, True, False]]
breakage_mask:  [[False, False, False]]
Interpretation: C and O may form a single or double bond, but no bond exists to break.


### `get_gnn_observation`

The observation deliberately separates current message-passing edges from candidate action pairs. Before bond formation, `edge_index` has zero columns but `pair_index` has one column. The GNN can therefore score a nonexistent C/O bond without pretending it is already a message-passing edge.

In [29]:
simple_observation = simple_env.get_gnn_observation()
show_shapes(simple_observation)
print("\nnode features x =\n", simple_observation["x"])
print("edge_index =\n", simple_observation["edge_index"])
print("pair_index =\n", simple_observation["pair_index"])
print("component_index =", simple_observation["component_index"].tolist())

  atom_indices           shape=(2,) dtype=torch.int64
  x                      shape=(2, 4) dtype=torch.float32
  edge_index             shape=(2, 0) dtype=torch.int64
  edge_attr              shape=(0, 3) dtype=torch.float32
  pair_index             shape=(2, 1) dtype=torch.int64
  current_bond_orders    shape=(1,) dtype=torch.int64
  maximum_bond_orders    shape=(1,) dtype=torch.int64
  formation_mask         shape=(1, 3) dtype=torch.bool
  breakage_mask          shape=(1, 3) dtype=torch.bool
  molecule_features      shape=(2, 15) dtype=torch.float32
  component_index        shape=(2,) dtype=torch.int64

node features x =
 tensor([[1., 0., 0., 4.],
        [0., 0., 1., 2.]])
edge_index =
 tensor([], size=(2, 0), dtype=torch.int64)
pair_index =
 tensor([[0],
        [1]])
component_index = [0, 1]


### Query and validation helpers

The next cell demonstrates `molecule_containing`, `_current_bond_order`, `_maximum_reachable_order`, `_bond_key`, `_bond_sort_key`, `_minimum_atom_index`, `_require_registered_atom`, and `_validate_bond_change`. The underscore helpers are shown for understanding; application code normally does not need to call them.

In [30]:
print("molecule containing carbon:",
      tuple(a.index for a in simple_env.molecule_containing(carbon).atoms))
print("_current_bond_order(C, O):",
      simple_env._current_bond_order(carbon, oxygen))
print("_maximum_reachable_order(C, O, 0):",
      simple_env._maximum_reachable_order(carbon, oxygen, 0))
print("_bond_key(1, 0):", simple_env._bond_key(1, 0))
print("_minimum_atom_index(oxygen singleton):",
      simple_env._minimum_atom_index(oxygen_species))

simple_env._require_registered_atom(carbon)       # succeeds silently
simple_env._validate_bond_change(2)               # succeeds silently
print("registered carbon and bond_change=2 were accepted")

show_expected_error(
    "A new atom object reusing index 0 is not the registered carbon",
    lambda: simple_env._require_registered_atom(atom(0, "C")),
)
show_expected_error(
    "A zero bond change is rejected",
    lambda: simple_env._validate_bond_change(0),
)

molecule containing carbon: (0,)
_current_bond_order(C, O): 0
_maximum_reachable_order(C, O, 0): 2
_bond_key(1, 0): (0, 1)
_minimum_atom_index(oxygen singleton): 1
registered carbon and bond_change=2 were accepted
A new atom object reusing index 0 is not the registered carbon: ValueError: Atom 0 is not registered in this oven.
A zero bond change is rejected: ValueError: bond_change must be positive.


### `bond_formation`, `_merge_molecules`, `_replace_molecules`, and `_update_pair_tables`

We directly request `bond_change=2`. `bond_formation` validates the endpoints and change, consumes two remaining valences from each endpoint, creates a double bond, and detects that the atoms were in different components. `_merge_molecules` constructs a connected C=O snapshot. `_replace_molecules` removes the two stale singleton snapshots, inserts the merged snapshot, and invokes `_rebuild_atom_to_molecule`. Finally, `_update_pair_tables` refreshes the selected current order and every maximum-order row incident to either endpoint.

In [31]:
formed_bond = simple_env.bond_formation(carbon, oxygen, bond_change=2)
show_state(simple_env, "After bond_formation(C, O, bond_change=2)")
print("returned Bond order:", formed_bond.order)
print("sorted bond keys using _bond_sort_key:",
      [bond.node_indices for bond in sorted(simple_env.bonds.values(),
                                             key=simple_env._bond_sort_key)])
print("carbon and oxygen now map to the same snapshot:",
      simple_env.molecule_containing(carbon) is simple_env.molecule_containing(oxygen))


After bond_formation(C, O, bond_change=2)
  components: [(0, 1)]
  bonds: {(0, 1): 2}
  remaining valence: {0: 2, 1: 0}
  pair tables:
    {'pair': (0, 1), 'current_BO': 2, 'maximum_BO': 2}
returned Bond order: 2
sorted bond keys using _bond_sort_key: [(0, 1)]
carbon and oxygen now map to the same snapshot: True


### `bond_breakage`: partial decrease

Reducing C=O by one produces C–O. Because a positive bond order remains, connectivity cannot change. The old molecule snapshot is replaced with a new snapshot containing a single bond, but the number of molecules stays one.

In [32]:
weakened_bond = simple_env.bond_breakage(carbon, oxygen, bond_change=1)
show_state(simple_env, "After partial bond_breakage(C, O, 1)")
print("returned replacement Bond order:", weakened_bond.order)
print("The pair now allows formation by 1 and breakage by 1:",
      simple_env.get_valid_actions())


After partial bond_breakage(C, O, 1)
  components: [(0, 1)]
  bonds: {(0, 1): 1}
  remaining valence: {0: 3, 1: 1}
  pair tables:
    {'pair': (0, 1), 'current_BO': 1, 'maximum_BO': 2}
returned replacement Bond order: 1
The pair now allows formation by 1 and breakage by 1: {(0, 1, 'formation'): 1, (0, 1, 'breakage'): 1}


### `bond_breakage` and `_split_molecule`: complete removal

The second decrease removes the remaining C–O bond. `_split_molecule` computes connected components using the remaining bonds. Here it finds two components, so `_replace_molecules` installs two singleton molecule snapshots. The bond action returns `None` because no replacement bond remains.

In [33]:
removed_bond = simple_env.bond_breakage(carbon, oxygen, bond_change=1)
show_state(simple_env, "After complete removal of the C-O bond")
print("return value:", removed_bond)
print("molecule count:", len(simple_env.molecules))


After complete removal of the C-O bond
  components: [(0,), (1,)]
  bonds: {}
  remaining valence: {0: 4, 1: 2}
  pair tables:
    {'pair': (0, 1), 'current_BO': 0, 'maximum_BO': 2}
return value: None
molecule count: 2


### `validate_state`

`validate_state` is a diagnostic invariant check. It verifies molecule connectivity, exact atom ownership, bond consistency, valence accounting, fixed pair rows, both bond-order tables, and the valid-action cache. A return value of `True` means all checks passed; an inconsistency raises an informative exception.

In [34]:
print("simple_env.validate_state() ->", simple_env.validate_state())

simple_env.validate_state() -> True


### `step`

Training code will usually use `step` instead of calling a mutation method directly. An action is `(atom_i, atom_j, action_type, bond_change)`. The changing of the current environment state still performs the chemical safety checks. `step` increments the episode counter and returns `(observation, reward, terminated, truncated, info)`. Reward is currently a placeholder `0.0`.

In [35]:
step_c_species = singleton(0, "C")
step_o_species = singleton(1, "O")
step_env = MoleculeEnv([step_c_species, step_o_species], max_steps=3)

transition = step_env.step((0, 1, MoleculeEnv.FORMATION, 1))
observation, reward, terminated, truncated, info = transition
print("reward:", reward)
print("terminated:", terminated, "truncated:", truncated)
print("info:", info)
print("components after step:", component_sets(step_env))

reward: 0.0
terminated: False truncated: False
info: {'action_type': 'formation', 'bond_change': 1, 'old_bond_order': 0, 'new_bond_order': 1, 'molecule_count_before': 2, 'molecule_count_after': 1, 'merged': True, 'split': False}
components after step: [(0, 1)]


# Part 2 — Complex carbon-chain/oxygen example

The initial carbon component is the chain `10–11–12–13`, and atom 14 is an oxygen singleton. With five atoms, the fixed candidate table has `P = 5×4/2 = 10` unordered rows.

The sequence will demonstrate two important connectivity cases:

- removing a bond from a ring can leave the molecule connected;
- removing a bridge breaks one molecule into two.

In [42]:
carbon_chain, complex_atoms = connected_molecule(
    atom_specs=[(10, "C"), (11, "C"), (12, "C"), (13, "C")],
    bond_specs=[(10, 11, 1), (11, 12, 1), (12, 13, 1)],
)
print(complex_atoms)
oxygen_singleton = singleton(14, "O")
# complex_atoms[14] = oxygen_singleton.atoms[0]   # Add the oxygen atom to the complex_atoms dictionary

complex_env = MoleculeEnv([carbon_chain, oxygen_singleton], max_steps=20)
show_state(complex_env, "Complex initial state")
print("number of candidate rows:", len(complex_env.pair_index))
print("initial state is valid:", complex_env.validate_state())

{10: <obj_node.atom object at 0x12e47cf10>, 11: <obj_node.atom object at 0x12e4bb350>, 12: <obj_node.atom object at 0x12e4cd6d0>, 13: <obj_node.atom object at 0x12e4cd650>}

Complex initial state
  components: [(10, 11, 12, 13), (14,)]
  bonds: {(10, 11): 1, (11, 12): 1, (12, 13): 1}
  remaining valence: {10: 3, 11: 2, 12: 2, 13: 3, 14: 2}
  pair tables:
    {'pair': (10, 11), 'current_BO': 1, 'maximum_BO': 3}
    {'pair': (10, 12), 'current_BO': 0, 'maximum_BO': 2}
    {'pair': (10, 13), 'current_BO': 0, 'maximum_BO': 3}
    {'pair': (10, 14), 'current_BO': 0, 'maximum_BO': 2}
    {'pair': (11, 12), 'current_BO': 1, 'maximum_BO': 3}
    {'pair': (11, 13), 'current_BO': 0, 'maximum_BO': 2}
    {'pair': (11, 14), 'current_BO': 0, 'maximum_BO': 2}
    {'pair': (12, 13), 'current_BO': 1, 'maximum_BO': 3}
    {'pair': (12, 14), 'current_BO': 0, 'maximum_BO': 2}
    {'pair': (13, 14), 'current_BO': 0, 'maximum_BO': 2}
number of candidate rows: 10
initial state is valid: True


## Complex transition 1 — internal formation closes a ring

Forming bond `(10, 13)` occurs inside one molecule, so `_merge_molecules` is not needed. `bond_formation` creates a four-carbon cycle and `_replace_molecules` performs a one-snapshot-to-one-snapshot replacement. The molecule count remains two because oxygen is still separate.

In [43]:
_, _, _, _, ring_info = complex_env.step(
    (10, 13, MoleculeEnv.FORMATION, 1)
)
show_state(complex_env, "After closing the four-carbon ring")
print("step info:", ring_info)
print("atom 10 component:",
      tuple(a.index for a in complex_env.molecule_containing(complex_atoms[10]).atoms))


After closing the four-carbon ring
  components: [(10, 11, 12, 13), (14,)]
  bonds: {(10, 11): 1, (10, 13): 1, (11, 12): 1, (12, 13): 1}
  remaining valence: {10: 2, 11: 2, 12: 2, 13: 2, 14: 2}
  pair tables:
    {'pair': (10, 11), 'current_BO': 1, 'maximum_BO': 3}
    {'pair': (10, 12), 'current_BO': 0, 'maximum_BO': 2}
    {'pair': (10, 13), 'current_BO': 1, 'maximum_BO': 3}
    {'pair': (10, 14), 'current_BO': 0, 'maximum_BO': 2}
    {'pair': (11, 12), 'current_BO': 1, 'maximum_BO': 3}
    {'pair': (11, 13), 'current_BO': 0, 'maximum_BO': 2}
    {'pair': (11, 14), 'current_BO': 0, 'maximum_BO': 2}
    {'pair': (12, 13), 'current_BO': 1, 'maximum_BO': 3}
    {'pair': (12, 14), 'current_BO': 0, 'maximum_BO': 2}
    {'pair': (13, 14), 'current_BO': 0, 'maximum_BO': 2}
step info: {'action_type': 'formation', 'bond_change': 1, 'old_bond_order': 0, 'new_bond_order': 1, 'molecule_count_before': 2, 'molecule_count_after': 2, 'merged': False, 'split': False}
atom 10 component: (10, 11, 12, 

## Complex transition 2 — complete removal of a non-bridge bond

We completely remove `(11, 12)`. `_split_molecule` must still run because only a graph search can determine whether connectivity changed. The alternative path `11–10–13–12` remains, so it finds one connected component. The molecule count therefore stays two. This is why complete bond removal cannot automatically be treated as a molecule split.

In [44]:
_, _, _, _, non_bridge_info = complex_env.step(
    (11, 12, MoleculeEnv.BREAKAGE, 1)
)
show_state(complex_env, "After removing non-bridge bond (11, 12)")
print("step info:", non_bridge_info)
print("split flag is false:", non_bridge_info["split"] is False)


After removing non-bridge bond (11, 12)
  components: [(10, 11, 12, 13), (14,)]
  bonds: {(10, 11): 1, (10, 13): 1, (12, 13): 1}
  remaining valence: {10: 2, 11: 3, 12: 3, 13: 2, 14: 2}
  pair tables:
    {'pair': (10, 11), 'current_BO': 1, 'maximum_BO': 3}
    {'pair': (10, 12), 'current_BO': 0, 'maximum_BO': 2}
    {'pair': (10, 13), 'current_BO': 1, 'maximum_BO': 3}
    {'pair': (10, 14), 'current_BO': 0, 'maximum_BO': 2}
    {'pair': (11, 12), 'current_BO': 0, 'maximum_BO': 3}
    {'pair': (11, 13), 'current_BO': 0, 'maximum_BO': 2}
    {'pair': (11, 14), 'current_BO': 0, 'maximum_BO': 2}
    {'pair': (12, 13), 'current_BO': 1, 'maximum_BO': 3}
    {'pair': (12, 14), 'current_BO': 0, 'maximum_BO': 2}
    {'pair': (13, 14), 'current_BO': 0, 'maximum_BO': 2}
step info: {'action_type': 'breakage', 'bond_change': 1, 'old_bond_order': 1, 'new_bond_order': 0, 'molecule_count_before': 2, 'molecule_count_after': 2, 'merged': False, 'split': False}
split flag is false: True


## Complex transition 3 — double-bond formation merges two molecules

After removing `(11, 12)`, carbon 12 has enough remaining valence to form a double bond with oxygen 14. The requested change is 2, illustrating that one action need not change bond order by only one. `_merge_molecules` combines the carbon component and oxygen singleton, and the step reports `merged=True`.

In [45]:
pair_12_14 = complex_env._bond_key(12, 14)
row_12_14 = complex_env.pair_to_row[pair_12_14]
print("Before formation, pair row:", pair_rows(complex_env)[row_12_14])

_, _, _, _, merge_info = complex_env.step(
    (12, 14, MoleculeEnv.FORMATION, 2)
)
show_state(complex_env, "After forming C12=O14")
print("step info:", merge_info)
print("all atoms now share one molecule:", len(complex_env.molecules) == 1)

Before formation, pair row: {'pair': (12, 14), 'current_BO': 0, 'maximum_BO': 2}

After forming C12=O14
  components: [(10, 11, 12, 13, 14)]
  bonds: {(10, 11): 1, (10, 13): 1, (12, 13): 1, (12, 14): 2}
  remaining valence: {10: 2, 11: 3, 12: 1, 13: 2, 14: 0}
  pair tables:
    {'pair': (10, 11), 'current_BO': 1, 'maximum_BO': 3}
    {'pair': (10, 12), 'current_BO': 0, 'maximum_BO': 1}
    {'pair': (10, 13), 'current_BO': 1, 'maximum_BO': 3}
    {'pair': (10, 14), 'current_BO': 0, 'maximum_BO': 0}
    {'pair': (11, 12), 'current_BO': 0, 'maximum_BO': 1}
    {'pair': (11, 13), 'current_BO': 0, 'maximum_BO': 2}
    {'pair': (11, 14), 'current_BO': 0, 'maximum_BO': 0}
    {'pair': (12, 13), 'current_BO': 1, 'maximum_BO': 2}
    {'pair': (12, 14), 'current_BO': 2, 'maximum_BO': 2}
    {'pair': (13, 14), 'current_BO': 0, 'maximum_BO': 0}
step info: {'action_type': 'formation', 'bond_change': 2, 'old_bond_order': 0, 'new_bond_order': 2, 'molecule_count_before': 2, 'molecule_count_after': 1, 

## Complex transition 4 — bridge removal causes a true split

At this point `(10, 13)` is the only connection between the left fragment `{10, 11}` and the right fragment `{12, 13, 14}`. Removing it makes `_split_molecule` find two connected components. `_replace_molecules` installs both snapshots and `_rebuild_atom_to_molecule` assigns every atom to the correct new component.

In [46]:
_, _, _, _, split_info = complex_env.step(
    (10, 13, MoleculeEnv.BREAKAGE, 1)
)
show_state(complex_env, "After removing bridge bond (10, 13)")
print("step info:", split_info)
component_10 = complex_env.molecule_containing(complex_env.atoms[10])
component_14 = complex_env.molecule_containing(complex_env.atoms[14])
print("molecule containing atom 10:",
      tuple(a.index for a in component_10.atoms))
print("molecule containing atom 14:",
      tuple(a.index for a in component_14.atoms))


After removing bridge bond (10, 13)
  components: [(10, 11), (12, 13, 14)]
  bonds: {(10, 11): 1, (12, 13): 1, (12, 14): 2}
  remaining valence: {10: 3, 11: 3, 12: 1, 13: 3, 14: 0}
  pair tables:
    {'pair': (10, 11), 'current_BO': 1, 'maximum_BO': 3}
    {'pair': (10, 12), 'current_BO': 0, 'maximum_BO': 1}
    {'pair': (10, 13), 'current_BO': 0, 'maximum_BO': 3}
    {'pair': (10, 14), 'current_BO': 0, 'maximum_BO': 0}
    {'pair': (11, 12), 'current_BO': 0, 'maximum_BO': 1}
    {'pair': (11, 13), 'current_BO': 0, 'maximum_BO': 3}
    {'pair': (11, 14), 'current_BO': 0, 'maximum_BO': 0}
    {'pair': (12, 13), 'current_BO': 1, 'maximum_BO': 2}
    {'pair': (12, 14), 'current_BO': 2, 'maximum_BO': 2}
    {'pair': (13, 14), 'current_BO': 0, 'maximum_BO': 0}
step info: {'action_type': 'breakage', 'bond_change': 1, 'old_bond_order': 1, 'new_bond_order': 0, 'molecule_count_before': 1, 'molecule_count_after': 2, 'merged': False, 'split': True}
molecule containing atom 10: (10, 11)
molecule 

## Inspect the final complex GNN observation

The final observation shows how all views remain aligned after several edits. In particular, `component_index` maps each node row to a molecule-feature row, while `pair_index` still has the same ten columns it had at initialization. `edge_index` contains only the three bonds that currently exist and stores both directions for message passing.

In [47]:
complex_observation = complex_env.get_gnn_observation()
show_shapes(complex_observation)
print("\natom_indices:", complex_observation["atom_indices"].tolist())
print("component_index:", complex_observation["component_index"].tolist())
print("molecule feature rows:", complex_observation["molecule_features"].shape[0])
print("current pair orders:", complex_observation["current_bond_orders"].tolist())
print("maximum pair orders:", complex_observation["maximum_bond_orders"].tolist())
print("final state is valid:", complex_env.validate_state())

  atom_indices           shape=(5,) dtype=torch.int64
  x                      shape=(5, 4) dtype=torch.float32
  edge_index             shape=(2, 6) dtype=torch.int64
  edge_attr              shape=(6, 3) dtype=torch.float32
  pair_index             shape=(2, 10) dtype=torch.int64
  current_bond_orders    shape=(10,) dtype=torch.int64
  maximum_bond_orders    shape=(10,) dtype=torch.int64
  formation_mask         shape=(10, 3) dtype=torch.bool
  breakage_mask          shape=(10, 3) dtype=torch.bool
  molecule_features      shape=(2, 15) dtype=torch.float32
  component_index        shape=(5,) dtype=torch.int64

atom_indices: [10, 11, 12, 13, 14]
component_index: [0, 0, 1, 1, 1]
molecule feature rows: 2
current pair orders: [1, 0, 0, 0, 0, 0, 0, 1, 2, 0]
maximum pair orders: [3, 1, 3, 0, 1, 3, 0, 2, 2, 0]
final state is valid: True


## How `_update_pair_tables` limits its work

After an action on atoms `i` and `j`, only one `current_BO` entry can change: pair `(i, j)`. However, the remaining valence of both endpoints changes, so the maximum reachable order may change for every pair incident to `i` or `j`. `_update_pair_tables` updates exactly those rows rather than rebuilding all pair data. Rows not touching either endpoint are unchanged.

`get_valid_actions` then reads the tables and refreshes the action cache. This keeps pair data (edge/action information) separate from molecule snapshots (subgraph information).

In [49]:
final_actions = complex_env.get_valid_actions()
final_formation_mask, final_breakage_mask = complex_env.get_valid_action_masks()
print("number of cached action for the next step:", len(final_actions))
print("formation choices allowed:", int(final_formation_mask.sum()))
print("breakage choices allowed:", int(final_breakage_mask.sum()))
print("sorted existing bonds:",
      [(bond.node_indices, bond.order)
       for bond in sorted(complex_env.bonds.values(), key=complex_env._bond_sort_key)])

number of cached action for the next step: 9
formation choices allowed: 11
breakage choices allowed: 4
sorted existing bonds: [((10, 11), 1), ((12, 13), 1), ((12, 14), 2)]


# Method-by-method takeaways

### Methods normally called by the training loop

- `step(action)` is the main transition interface.
- `get_gnn_observation()` constructs the model input.
- `get_valid_actions()` is useful when a dictionary of maximum changes is convenient.
- `get_valid_action_masks()` is useful for masking action logits.
- `validate_state()` is especially useful in tests and debugging.

### Mutation methods

- `bond_formation()` and `bond_breakage()` contain the local chemical checks and graph mutation logic. `step()` delegates to them.

### Query method

- `molecule_containing(atom)` retrieves current subgraph membership. Because molecule objects are replaced after edits, do not keep an old molecule snapshot and assume it remains current.

### Internal maintenance methods

- `_replace_molecules`, `_merge_molecules`, `_split_molecule`, and `_rebuild_atom_to_molecule` keep connected-component snapshots synchronized.
- `_build_pair_tables` and `_update_pair_tables` maintain the fixed candidate-pair representation.
- `_current_bond_order` and `_maximum_reachable_order` compute the two bond-order table values.
- `_require_registered_atom` and `_validate_bond_change` protect mutations.
- `_bond_key`, `_bond_sort_key`, and `_minimum_atom_index` make unordered-pair keys and output ordering deterministic.

Together, these methods ensure that node features, current edges, candidate actions, and molecule features all describe the same reaction state after every action.

## Suggested experiments

Try changing one item at a time and inspect the tables and masks:

1. Form `(10, 13)` with a larger bond change and observe which later actions disappear as valence is consumed.
2. Break the C=O bond by 1 instead of 2 and confirm that the molecule does not split while a single bond remains.
3. Set `max_steps=2` and observe `truncated=True` on the second successful call to `step`.
4. Pass a bond change larger than the value in `get_valid_actions()` and inspect the safety error raised by the mutation method.
5. Compare `edge_index.shape[1]` with `pair_index.shape[1]` as bonds form and break: the first changes, while the second remains fixed.